# 04 — POMDP + belief state using Elder Mist

You already know:

```text
state → policy → action → transition
```

Now make the world **partially observable**.

The true world contains:

```text
Sword active = YES/NO
```

but our agent gets only a noisy sensor:

```text
"glint detected"
or
"no glint"
```

That makes the agent maintain a **belief** about the hidden Sword state.

A belief state is not “the true state.”

It is:

> what I currently believe about hidden state, given observations/history.


In [ ]:
from pathlib import Path
import sys, random, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo = Path.cwd()
if repo.name == "notebooks":
    repo = repo.parent
sys.path.insert(0, str(repo))

from eldermist import ElderMistReplicaEnv


## 1. Hidden truth vs observation

Assume the sensor behaves like this:

```text
P(glint | Sword active) = 0.85
P(glint | Sword down)   = 0.10
```

So a glint is evidence, not certainty.


In [ ]:
P_GLINT_IF_ACTIVE = 0.85
P_GLINT_IF_DOWN = 0.10

def observe_glint(true_active, rng):
    p = P_GLINT_IF_ACTIVE if true_active else P_GLINT_IF_DOWN
    return rng.random() < p

def bayes_update(prior_active, glint):
    if glint:
        likelihood_active = P_GLINT_IF_ACTIVE
        likelihood_down = P_GLINT_IF_DOWN
    else:
        likelihood_active = 1 - P_GLINT_IF_ACTIVE
        likelihood_down = 1 - P_GLINT_IF_DOWN

    numerator = likelihood_active * prior_active
    denominator = numerator + likelihood_down * (1 - prior_active)
    return numerator / denominator

for prior in [0.2, 0.5, 0.8]:
    print(
        "prior", prior,
        "-> glint posterior", round(bayes_update(prior, True), 3),
        "| no-glint posterior", round(bayes_update(prior, False), 3)
    )


Read one line literally:

```text
prior P(Sword active) = 0.50
observe glint
posterior P(Sword active) ≈ ...
```

The observation changed **your belief**.

It did not change the Sword.


In [ ]:
rng = random.Random(4)
true_active = True
belief = 0.50
rows = []

for t in range(8):
    glint = observe_glint(true_active, rng)
    belief = bayes_update(belief, glint)
    rows.append({
        "t": t,
        "true_sword_active": true_active,
        "observation_glint": glint,
        "belief_P_active": belief,
    })

pd.DataFrame(rows)


In [ ]:
df = pd.DataFrame(rows)
plt.figure(figsize=(8,4))
plt.plot(df["t"], df["belief_P_active"], marker="o")
plt.axhline(1.0 if true_active else 0.0, linestyle="--", label="hidden truth")
plt.ylim(0,1)
plt.xlabel("Observation number")
plt.ylabel("Belief P(Sword active)")
plt.title("Belief changes as noisy observations arrive")
plt.legend()
plt.show()


## 2. Why history matters

If you only look at the latest sensor reading:

```text
no glint
```

you might say “Sword is down.”

But history might be:

```text
glint
glint
glint
no glint
```

Your posterior may still strongly favor Sword active.

So in a POMDP:

```text
latest observation ≠ sufficient state
```

History can help construct a sufficient internal representation.

That internal representation can be:

- an explicit probability distribution like this notebook,
- a recurrent-network hidden state,
- an LLM summary of history,
- a learned latent vector,
- some hybrid.


## 3. Connect this to LifeOS / employee cloning

Hidden world might include:

```text
Carl's real current objective
whether an external application was approved but not yet emailed
whether a coworker is about to interrupt
what task dependency is unresolved
```

Observations might be:

```text
screen
calendar
Slack
keyboard
camera
previous actions
```

Your state representation is an attempt to summarize what matters.

A **belief state** becomes useful when that summary explicitly represents uncertainty about hidden facts.


### Sentence to memorize

```text
Observation = what I just saw.
History     = what I have seen/done so far.
Belief      = what I think the hidden world probably is, given that history.
```
